In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/recheckaspectdataset/udemy_comments_with_aspects_chunk2.csv
/kaggle/input/recheckaspectdataset/udemy_comments_with_aspects_chunk1.csv
/kaggle/input/recheckaspectdataset/udemy_comments_with_aspects_chunk3.csv
/kaggle/input/bestelectrafulldataset/config.json
/kaggle/input/bestelectrafulldataset/training_args.bin
/kaggle/input/bestelectrafulldataset/model.safetensors
/kaggle/input/bestrobertamodelfulldataset/config.json
/kaggle/input/bestrobertamodelfulldataset/training_args.bin
/kaggle/input/bestrobertamodelfulldataset/model.safetensors
/kaggle/input/commentswithaspectchunk/udemy_comments_with_aspects_chunk2.csv
/kaggle/input/commentswithaspectchunk/udemy_comments_with_aspects_chunk1.csv
/kaggle/input/commentswithaspectchunk/udemy_comments_with_aspects_chunk3.csv
/kaggle/input/bestxlnetmodelonfulldataset/config.json
/kaggle/input/bestxlnetmodelonfulldataset/training_args.bin
/kaggle/input/bestxlnetmodelonfulldataset/model.safetensors


In [2]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm import tqdm
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_configs = [
    ("/kaggle/input/bestelectrafulldataset", "google/electra-base-discriminator", 64),
    ("/kaggle/input/bestrobertamodelfulldataset", "roberta-base", 128),
    ("/kaggle/input/bestxlnetmodelonfulldataset", "xlnet-base-cased", 128)
]

models = []
tokenizers = []
max_lengths = []
for model_path, base_name, max_length in model_configs:
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()
    models.append(model)
    tokenizers.append(AutoTokenizer.from_pretrained(base_name))
    max_lengths.append(max_length)

input_file = "/kaggle/input/recheckaspectdataset/udemy_comments_with_aspects_chunk1.csv"
df = pd.read_csv(input_file)

chunk_match = re.search(r'chunk(\d+)', input_file)
chunk_number = chunk_match.group(1) if chunk_match else "unknown"

df_original_size = len(df)
df = df[df['predicted_aspect'] != 'Ambiguous']
df_new_size = len(df)
print(f"Removed {df_original_size - df_new_size} rows with Ambiguous predicted_aspect")

def predict(text, model, tokenizer, max_length):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=max_length).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    return probabilities.squeeze().cpu().numpy()

def ensemble_predict(text, models, tokenizers, max_lengths):
    predictions = [predict(text, model, tokenizer, max_length) 
                   for model, tokenizer, max_length in zip(models, tokenizers, max_lengths)]
    avg_probs = np.mean(predictions, axis=0)
    return np.argmax(avg_probs)

predictions = []
for sentence in tqdm(df['sentence'], desc="Predicting sentiments"):
    prediction = ensemble_predict(sentence, models, tokenizers, max_lengths)
    predictions.append(prediction)

sentiment_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
df['predicted_sentiment_ensemble'] = [sentiment_map[pred] for pred in predictions]

output_filename = f'ensemble_predictions_chunk{chunk_number}_full_custom_max_length_no_ambiguous.csv'
df.to_csv(output_filename, index=False)
print(f"Predictions completed and saved to '{output_filename}'")

print(df[['sentence', 'predicted_sentiment_ensemble', 'predicted_aspect'] + [col for col in df.columns if col not in ['sentence', 'predicted_sentiment_ensemble', 'predicted_aspect']]].head())

if 'predicted_sentiment' in df.columns:
    accuracy = (df['predicted_sentiment'] == df['predicted_sentiment_ensemble']).mean()
    print(f"Accuracy of ensemble predictions compared to original predictions: {accuracy:.4f}")

print("\nDistribution of ensemble predictions:")
print(df['predicted_sentiment_ensemble'].value_counts(normalize=True))

print("\nDistribution of aspects after removing Ambiguous:")
print(df['predicted_aspect'].value_counts(normalize=True))

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

Removed 775351 rows with Ambiguous predicted_aspect


Predicting sentiments: 100%|██████████| 342769/342769 [3:57:10<00:00, 24.09it/s]


Predictions completed and saved to 'ensemble_predictions_chunk1_full_custom_max_length_no_ambiguous.csv'
                                             sentence  \
3   Aviva is such a natural teacher and healer/hea...   
4   This was really cool opportunity to witness a ...   
7   This breakdance instructor breakdance with suc...   
11  It was full of information, well structured an...   
13                       Very unique way of teaching.   

   predicted_sentiment_ensemble       predicted_aspect  \
3                      Positive             Instructor   
4                      Positive  Assignment & Tutorial   
7                      Positive             Instructor   
11                     Positive      Design & Material   
13                     Positive             Instructor   

                        group_category  \
3   Personal Development and Lifestyle   
4   Personal Development and Lifestyle   
7                   Health and Fitness   
11          Marketing and Productiv